In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import current_timestamp, col

# 1. Define paths based on your UC Volume
volume_path = "/Volumes/railway_analytics/bronze/raw_landing/"
live_snapshot_path = f"{volume_path}live_train_snapshot_20260523.csv"
scd2_history_path = f"{volume_path}scd2_bootstrap_history.csv"

# 2. Load and Append Live Train Status (Bronze)
print("Loading Live Train Snapshot...")

# FIX: Remove inferSchema. Everything loads as STRING by default.
live_df = spark.read.format("csv") \
    .option("header", "true") \
    .load(live_snapshot_path)

# Cast only the specific column that is defined as INT in your DDL
if "delay_mins" in live_df.columns:
    live_df = live_df.withColumn("delay_mins", col("delay_mins").cast("int"))

# Ensure ingested_at timestamp is present
if "ingested_at" not in live_df.columns:
    live_df = live_df.withColumn("ingested_at", current_timestamp())

# Append to the managed table
live_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("bronze.live_train_status")
print("Successfully loaded bronze.live_train_status")

# 3. Load and Append SCD2 Bootstrap History (Gold)
print("Loading SCD2 Bootstrap History...")

# FIX: Remove inferSchema here as well.
scd2_df = spark.read.format("csv") \
    .option("header", "true") \
    .load(scd2_history_path)

# Cast the specific numeric and date columns to match the Gold DDL
if "route_stops" in scd2_df.columns:
    scd2_df = scd2_df.withColumn("route_stops", col("route_stops").cast("int"))
if "distance_km" in scd2_df.columns:
    scd2_df = scd2_df.withColumn("distance_km", col("distance_km").cast("int"))
if "journey_duration_h" in scd2_df.columns:
    scd2_df = scd2_df.withColumn("journey_duration_h", col("journey_duration_h").cast("double"))
if "eff_from_date" in scd2_df.columns:
    scd2_df = scd2_df.withColumn("eff_from_date", col("eff_from_date").cast("date"))
if "eff_to_date" in scd2_df.columns:
    scd2_df = scd2_df.withColumn("eff_to_date", col("eff_to_date").cast("date"))

# Append to the managed table
scd2_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("gold.dim_train_schedule_scd2")
print("Successfully loaded gold.dim_train_schedule_scd2")

Loading Live Train Snapshot...
Successfully loaded bronze.live_train_status
Loading SCD2 Bootstrap History...
Successfully loaded gold.dim_train_schedule_scd2
